In [ ]:
import pandapower as pp
import pandapower.plotting as plot
import matplotlib.pyplot as plt

# 【核心指令执行】手动加载中文字体与高分辨率画布
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows常用中文字体，Mac用户可改为 'Arial Unicode MS'
plt.rcParams['axes.unicode_minus'] = False    # 正常显示负号
plt.figure(figsize=(8, 6), dpi=300)           # 强制高分辨率 300 DPI

net = pp.create_empty_network()

# 1. 创建节点
b1 = pp.create_bus(net, name="110kV 母线", vn_kv=110.0, geodata=(0, 2))
b2 = pp.create_bus(net, name="20kV 母线", vn_kv=20.0, geodata=(0, 0))

# 2. 创建变压器
pp.create_transformer_from_parameters(net, hv_bus=b1, lv_bus=b2, name="25MVA 变压器",
                                      sn_mva=25.0, vn_hv_kv=110.0, vn_lv_kv=20.0,
                                      vk_percent=10.0, vkr_percent=0.3, pfe_kw=20.0, i0_percent=0.1)

# 3. 负载与电源
pp.create_load(net, bus=b2, p_mw=100.0, q_mvar=50.0, name="100MW 重负载")
pp.create_ext_grid(net, bus=b1, vm_pu=1.0, name="外部主网")

# 4. 绘图：开启节点颜色标识和标签
print("正在生成高分辨率中文拓扑图...")
plot.simple_plot(net, 
                 bus_color="blue",       # 母线标蓝
                 trafo_color="red",      # 变压器连线标红
                 ext_grid_color="green", # 电源标绿
                 show_plot=False)

plt.title("110kV/20kV 降压供电系统拓扑 (高分辨率版)", fontsize=14)
plt.show()

In [ ]:
import pandapower as pp
import pandapower.plotting.plotly as pplot
# 确保你装了 plotly: pip install plotly

net = pp.create_empty_network()

# 1. 创建节点（Plotly会自动调整位置，或者手动调整geodata使其美观）
b1 = pp.create_bus(net, name="110kV Bus B1", vn_kv=110.0, geodata=(0, 2))
b2 = pp.create_bus(net, name="20kV Bus B2", vn_kv=20.0, geodata=(0, 0))

# 2. 创建变压器
pp.create_transformer_from_parameters(net, hv_bus=b1, lv_bus=b2, name="25MVA Trafo",
                                      sn_mva=25.0, vn_hv_kv=110.0, vn_lv_kv=20.0,
                                      vk_percent=10.0, vkr_percent=0.3, pfe_kw=20.0, i0_percent=0.1)

# 3. 负载与电源
pp.create_load(net, bus=b2, p_mw=100.0, q_mvar=50.0, name="Load 1")
pp.create_ext_grid(net, bus=b1, vm_pu=1.0, name="Ext Grid")

# 4. 初始化潮流（plotly绘图通常依赖结果，或者简单的简单绘图）
# pp.runpp(net) 

# 5. 使用 Plotly 绘制
print("正在生成 Plotly 交互式电气拓扑图...")
pplot.simple_plotly(net, respect_switches=True) # 这行代码会调用外部 Plotly 渲染器（通常弹出网页浏览器）

In [ ]:
import pandapower as pp
net = pp.create_empty_network()
b1 = pp.create_bus(net , vn_kv= 10)
b2 = pp.create_bus(net, vn_kv= 10)
pp.create_ext_grid(net, bus=b1, vm_pu=1.0, name="Ext Grid")
pp.create_load(net , bus = b2 , p_mw = 2. , name = 'Load1')
pp.create_line(net , from_bus= b1 , to_bus=b2 , name="Line1" , length_km=5.0 , std_type='NAYY 4x50 SE')
pp.runpp(net)
print(net.res_bus.p_mw)
print(net.res_line.pl_mw)

In [ ]:
net = pp.create_empty_network()
b1 = pp.create_bus(net, vn_kv= 10)
b2 = pp.create_bus(net, vn_kv= 10)
l1 = pp.create_line(net , from_bus= b1 , to_bus= b2 , length_km= 5. , std_type='NAYY 4x50 SE')
pp.create_ext_grid(net , bus = b1 , vm_pu=1. , name="Ext Grid")
pp.create_load(net , bus = b2 , p_mw = 2. , name = 'Load1')
sw1 = pp.create_switch(net , bus = b1 , element= l1 , et = 'l' , closed=True , name = '开关1')
pp.runpp(net)
print(net.res_line.i_ka.loc[l1])
net.switch.loc[sw1 , 'closed'] = False
try:
    pp.runpp(net)
    print(net.res_line.i_ka.loc[l1])
except Exception as e:
    print('出错')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows常用黑体
plt.rcParams['axes.unicode_minus'] = False # 正常显示负号
net = pp.create_empty_network()
b1 = pp.create_bus(net, vn_kv= 10)
b2 = pp.create_bus(net, vn_kv= 10)
l1 = pp.create_line(net , from_bus= b1 , to_bus= b2 , length_km= 5. , std_type='NAYY 4x50 SE')
pp.create_ext_grid(net , bus = b1 , vm_pu=1. , name="Ext Grid")
pp.create_load(net , bus = b2 , p_mw = 2. , name = 'Load1')
sw1 = pp.create_switch(net , bus = b1 , element= l1 , et = 'l' , closed=True , name = '开关1')
test_load = np.arange(1. , 16. , 1.)
voltage_record = []
for p in test_load:
    net.load.loc[0 , 'p_mw'] = p
    try:
            pp.runpp(net)
            current_v = net.res_bus.loc[b2 , 'vm_pu']
            voltage_record.append(current_v)
            print(f'负荷{p} ， 节点电压{current_v}')
    except Exception as e:
        print(f'负荷达到{p}mw，时候 电路负载过度 开始发散')
plt.figure(figsize=(8,6), dpi=300)
valid_load = test_load[:len(voltage_record)]
plt.plot(valid_load , voltage_record , marker='o' , c = 'r')
plt.grid(True)
plt.xlabel('负荷')
plt.ylabel('电压')
plt.show()

In [ ]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
net = pn.case33bw()
num_samples = 30000
dataset = []
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

for i in range(num_samples):
    noise_factor = np.random.uniform(0.8 , 1.2 , size=(len(net.load)))
    net.load['p_mw'] = base_p* noise_factor
    net.load['q_mvar'] = base_q * noise_factor
    try:
        pp.runpp(net , algorithm='nr')
        x_p = net.res_bus['p_mw'].values
        x_q = net.res_bus['q_mvar'].values
        y_vm = net.res_bus['vm_pu'].values
        y_va = net.res_bus['va_degree'].values
        sample = {}
        for bus_id in range(len(net.bus)):
            sample[f'bus_{bus_id}_P'] = x_p[bus_id]
            sample[f'bus_{bus_id}_Q'] = x_q[bus_id]
            sample[f'bus_{bus_id}_Vm'] = y_vm[bus_id]
            sample[f'bus_{bus_id}_Va'] = y_va[bus_id]
            
        dataset.append(sample)
    except Exception as e:
        pass
df = pd.DataFrame(dataset)
# 封箱！将列表转成二维表格
df = pd.DataFrame(dataset)
print(f"\n✅ 压铸完毕！成功提取有效样本: {len(df)} 条")

# ----------------- 终极保存动作 -----------------
save_path = "IEEE33_PyTorch_Dataset.csv"
# index=False 保证不会把 0,1,2,3 这种无意义的行号存入特征中
df.to_csv(save_path, index=False) 
print(f"📁 完美！数据已永久保存至你当前项目目录下的: {save_path}")
print("🎉 pandapower 的使命已彻底终结，是时候向 PyTorch 进军了！")

In [ ]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
import os
from tqdm.notebook import tqdm

net = pn.case33bw()
num_samples = 40000
csv_filename = "IEEE33_40000_Fast_Safe.csv"
chunk_size = 2000  # 卡车的容量：每攒够 2000 条刷入一次硬盘

base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

print(f"🚀 启动高速分块压铸机！目标 {num_samples} 条数据...")

# --- 1. 初始化空文件并写入表头 ---
headers = []
for bus_id in range(len(net.bus)):
    headers.extend([f'bus_{bus_id}_P', f'bus_{bus_id}_Q', f'bus_{bus_id}_Vm', f'bus_{bus_id}_Va'])

# 先建一个空表，只把表头写进去
pd.DataFrame(columns=headers).to_csv(csv_filename, index=False)

# --- 2. 启动循环 ---
batch_data = [] # 这就是你的“中转卡车”

for i in tqdm(range(num_samples), desc="造数进度"):
    noise_factor = np.random.uniform(0.8, 1.2, size=(len(net.load)))
    net.load['p_mw'] = base_p * noise_factor
    net.load['q_mvar'] = base_q * noise_factor
    
    try:
        pp.runpp(net, algorithm='nr')
        
        x_p = net.res_bus['p_mw'].values
        x_q = net.res_bus['q_mvar'].values
        y_vm = net.res_bus['vm_pu'].values
        y_va = net.res_bus['va_degree'].values
        
        row_data = []
        for bus_id in range(len(net.bus)):
            row_data.extend([x_p[bus_id], x_q[bus_id], y_vm[bus_id], y_va[bus_id]])
            
        # 把零件扔进卡车
        batch_data.append(row_data)
        
        # --- 3. 核心大招：满载发车 ---
        if len(batch_data) == chunk_size:
            # 打包成 DataFrame，以追加模式 ('a') 写入硬盘，不写表头 (header=False)
            pd.DataFrame(batch_data).to_csv(csv_filename, mode='a', header=False, index=False)
            # 发车后，立刻清空卡车，把内存释放掉！
            batch_data = [] 
            
    except Exception as e:
        pass

# --- 4. 扫尾工作 ---
# 循环结束后，卡车里可能还有没满 2000 的零散货物（比如最后剩下 500 条）
if len(batch_data) > 0:
    pd.DataFrame(batch_data).to_csv(csv_filename, mode='a', header=False, index=False)

print(f"🎉 完美收工！既没有撑爆内存，又保证了极致速度。文件位置：{os.path.abspath(csv_filename)}")

🚀 启动高速分块压铸机！目标 40000 条数据...


造数进度:   0%|          | 0/40000 [00:00<?, ?it/s]

In [1]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

# 1. 初始化标准 IEEE 33 节点网
net = pn.case33bw()
num_samples = 50000 
dataset = []

# 备份原始参数 (必须用 .values.copy() 彻底切断关联)
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# 预运行一次，开启缓存加速
pp.runpp(net)
recycle_settings = {'bus_pq': True, 'trafo_y': True, 'gen_p': True}

# 准备文件名
csv_filename = "IEEE33_50000_Safe_Base.csv"

print("🛡️ 安全防御模式已开启！")
print(f"💡 任务：单核压铸 {num_samples} 条数据。")
print("🚀 优势：鼠标丝滑、绝不黑屏、后台静默运行。")

# 2. 线性循环（就像老工匠雕花，慢但绝对不出错）
try:
    for i in tqdm(range(num_samples), desc="造数进度"):
        # 注入噪声 (80%-120%)
        noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
        net.load['p_mw'] = base_p * noise
        net.load['q_mvar'] = base_q * noise
        
        try:
            # 运行潮流计算
            pp.runpp(net, algorithm='nr', recycle=recycle_settings)
            
            # 提取结果
            res = net.res_bus
            x_p, x_q = res['p_mw'].values, res['q_mvar'].values
            y_vm, y_va = res['vm_pu'].values, res['va_degree'].values
            
            # 拼装 132 列数据
            row = []
            for b in range(33):
                row.extend([x_p[b], x_q[b], y_vm[b], y_va[b]])
            dataset.append(row)
            
            # 每 5000 条进行一次“存档”，防止意外
            if (i + 1) % 5000 == 0:
                temp_df = pd.DataFrame(dataset)
                temp_df.to_csv(csv_filename, index=False)
                # 打印一下进度，给你稳稳的安全感
                # print(f"\n📦 已自动存档至 {i+1} 条...")
                
        except:
            # 忽略极少数物理上不成立的崩溃点
            pass

    # 3. 最终大成
    final_df = pd.DataFrame(dataset)
    # 生成表头
    headers = []
    for i in range(33):
        headers.extend([f'bus_{i}_P', f'bus_{i}_Q', f'bus_{i}_Vm', f'bus_{i}_Va'])
    final_df.columns = headers
    
    final_df.to_csv(csv_filename, index=False)
    print(f"\n✅ 战役圆满成功！5万条弹药已入库：{csv_filename}")
    print(f"📊 最终尺寸: {final_df.shape}")

except KeyboardInterrupt:
    print("\n⚠️ 你手动停止了程序，正在保存已完成的部分...")
    if dataset:
        pd.DataFrame(dataset).to_csv(csv_filename, index=False)

🛡️ 安全防御模式已开启！
💡 任务：单核压铸 50000 条数据。
🚀 优势：鼠标丝滑、绝不黑屏、后台静默运行。


造数进度: 100%|██████████| 50000/50000 [00:13<00:00, 3608.63it/s]


ValueError: Length mismatch: Expected axis has 0 elements, new values have 132 elements

In [3]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm

net = pn.case33bw()
num_samples = 50000 
dataset = []

base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# 【关键改动 A】：先空跑一次，看环境稳不稳
print("🔍 正在进行系统自检...")
try:
    pp.runpp(net)
    print("✅ 基础潮流计算正常")
except Exception as e:
    print(f"❌ 基础环境就有问题: {e}")

print(f"🚀 开始压铸 50,000 条数据 (单线程稳健模式)...")

for i in tqdm(range(num_samples)):
    noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
    net.load['p_mw'] = base_p * noise
    net.load['q_mvar'] = base_q * noise
    
    try:
        # 【关键改动 B】：先去掉 recycle，保证 100% 兼容性
        pp.runpp(net, algorithm='nr')
        
        res = net.res_bus
        row = []
        for b in range(33):
            row.extend([res.at[b, 'p_mw'], res.at[b, 'q_mvar'], 
                        res.at[b, 'vm_pu'], res.at[b, 'va_degree']])
        dataset.append(row)
        
    except Exception as e:
        # 【关键改动 C】：如果是第一条就报错，立刻停下来告诉你原因
        if i == 0:
            print(f"\n💥 抓到真凶了！第一条数据就报错: {e}")
            break
        continue

# 3. 封箱保存
if len(dataset) > 0:
    headers = []
    for i in range(33):
        headers.extend([f'bus_{i}_P', f'bus_{i}_Q', f'bus_{i}_Vm', f'bus_{i}_Va'])
    
    final_df = pd.DataFrame(dataset, columns=headers)
    final_df.to_csv("IEEE33_50000_Safe_Final.csv", index=False)
    print(f"\n🎉 呼... 终于成功了！有效数据: {len(final_df)} 条")
else:
    print("\n😱 警告：一条数据都没存下来，请检查上面的报错信息！")

🔍 正在进行系统自检...
✅ 基础潮流计算正常
🚀 开始压铸 50,000 条数据 (单线程稳健模式)...


100%|██████████| 50000/50000 [18:37<00:00, 44.73it/s]



🎉 呼... 终于成功了！有效数据: 50000 条


In [1]:
import pandapower.networks as pn
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm

# 1. 初始化战场环境
net = pn.case33bw()
num_samples = 300 
dataset = []

# 提取基准负荷，作为波动的锚点
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()

# 2. 系统自检
print("🔍 正在进行系统自检...")
try:
    pp.runpp(net)
    print("✅ 基础潮流计算正常，准备全功率开动！")
except Exception as e:
    print(f"❌ 基础环境异常，请检查 pandapower 安装: {e}")
    exit()

print(f"🚀 开始压铸 300 条数据 (开启 init='flat' 绝对稳健模式)...")

# 3. 机器轰鸣：大循环开始
# 使用 tqdm 给你一个极其直观的进度条
for i in tqdm(range(num_samples), desc="数据生成进度"):
    # 注入高斯噪声：让负荷在 80% 到 120% 之间随机剧烈波动
    noise = np.random.uniform(0.8, 1.2, size=(len(net.load)))
    net.load['p_mw'] = base_p * noise
    net.load['q_mvar'] = base_q * noise
    
    try:
        # 【核心护城河】：init='flat' 强制每次重置电压，绝不吃上一次的残羹冷炙！
        pp.runpp(net, algorithm='nr', init='flat')
        
        # 如果收敛了，开始精准切割数据
        res = net.res_bus
        row = []
        for b in range(33):
            # 严格按照 P, Q, Vm, Va 的顺序打包
            row.extend([
                res.at[b, 'p_mw'], 
                res.at[b, 'q_mvar'], 
                res.at[b, 'vm_pu'], 
                res.at[b, 'va_degree']
            ])
        dataset.append(row)
        
    except Exception as e:
        # 电力系统的真实情况：极端波动下不收敛是正常的物理现象
        # 我们不需要让程序停下，直接无视并丢弃这个废点
        continue

# 4. 封箱贴条
if len(dataset) > 0:
    headers = []
    for i in range(33):
        headers.extend([f'bus_{i}_P', f'bus_{i}_Q', f'bus_{i}_Vm', f'bus_{i}_Va'])
    
    final_df = pd.DataFrame(dataset, columns=headers)
    final_df.to_csv("IEEE33_300_Safe_Final1.csv", index=False)
    
    # 打印最终战果：因为丢弃了不收敛的废点，最终数量可能会比 50000 稍微少几百个，这是最完美的真实状态！
    print(f"\n🎉 压铸完成！剔除极端报错点后，获得绝对纯净弹药: {len(final_df)} 条")
    print(f"📊 数据维度矩阵校验: {final_df.shape} (应为 行数 × 132列)")
else:
    print("\n😱 警告：全部计算崩溃，请检查参数设置！")

🔍 正在进行系统自检...
✅ 基础潮流计算正常，准备全功率开动！
🚀 开始压铸 300 条数据 (开启 init='flat' 绝对稳健模式)...


数据生成进度: 100%|██████████| 300/300 [00:01<00:00, 159.51it/s]



🎉 压铸完成！剔除极端报错点后，获得绝对纯净弹药: 300 条
📊 数据维度矩阵校验: (300, 132) (应为 行数 × 132列)
